# HPE Scratch Standalone (No Performance Evaluation)

Notebook ini self-contained: semua helper ada di dalam notebook.

Tujuan: model HPE dari awal tercipta dan bisa dicoba inferensi.

Tidak ada evaluasi OKS/PCK atau benchmarking latency.

Sel ini menyiapkan environment dasar, seed acak, dan perangkat komputasi yang dipakai untuk pelatihan dan inferensi.

In [ ]:
import json
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

Sel ini menjelaskan lokasi dataset dan parameter utama eksperimen, termasuk mode model serta ukuran input yang dipakai.

In [ ]:
# Ganti sesuai attach dataset Anda di Kaggle
DATA_ROOT = Path('/kaggle/input/datasets/yanplayz08/coco-subset-for-pose-estimation')
ANN_PATH = DATA_ROOT / 'annotations' / 'person_keypoints_train2017.json'
IMG_DIR = DATA_ROOT / 'train2017'

MODEL_KIND = 'topdown'   # pilih: topdown | bottomup | regresi

MAX_SAMPLES = 1800
EPOCHS = 2
BATCH_SIZE = 24
LR = 1e-3

IN_H, IN_W = 256, 192
HM_H, HM_W = 64, 48

In [ ]:
# Inferensi demo
# random  -> pakai sampel acak dari validation set
# index   -> pakai SAMPLE_IDX
# path    -> pakai SAMPLE_IMAGE_PATH
SAMPLE_MODE = 'random'
SAMPLE_IDX = 0
SAMPLE_IMAGE_PATH = None

# Jika ingin menggunakan gambar sendiri, ubah seperti di bawah.
# SAMPLE_MODE = 'path'
# SAMPLE_IMAGE_PATH = '/kaggle/input/.../gambar.jpg'

Sel ini memuat fungsi bantu untuk membaca data, membagi data latih-validasi, dan menyiapkan target heatmap atau koordinat.

In [ ]:
def load_samples(annotation_json, images_dir, max_samples=1800):
    """Membaca sampel pose COCO dan hanya mempertahankan anotasi orang yang valid.

    Mengembalikan daftar dict berisi image_path, bbox, dan keypoints 17x3.
    """
    with open(annotation_json, 'r', encoding='utf-8') as f:
        data = json.load(f)

    images_dir = Path(images_dir)
    image_by_id = {x['id']: x for x in data.get('images', [])}
    out = []

    for ann in data.get('annotations', []):
        k = ann.get('keypoints', [])
        if len(k) < 51:
            continue
        kpts = np.array(k, dtype=np.float32).reshape(17, 3)
        if int((kpts[:, 2] > 0).sum()) < 5:
            continue

        img_meta = image_by_id.get(ann.get('image_id'))
        if not img_meta:
            continue
        img_path = images_dir / img_meta['file_name']
        if not img_path.exists():
            continue

        bbox = np.array(ann.get('bbox', [0, 0, 0, 0]), dtype=np.float32)
        if bbox.shape[0] != 4 or bbox[2] <= 1 or bbox[3] <= 1:
            continue

        out.append({
            'image_path': str(img_path),
            'bbox': bbox,
            'kpts': kpts,
        })
        if len(out) >= max_samples:
            break
    return out


def split_samples(samples, val_ratio=0.2, seed=42):
    """Membagi sampel menjadi data latih dan validasi dengan seed tetap."""
    rng = random.Random(seed)
    idx = list(range(len(samples)))
    rng.shuffle(idx)
    n_val = int(len(idx) * val_ratio)
    val_ids = set(idx[:n_val])
    tr, va = [], []
    for i, s in enumerate(samples):
        (va if i in val_ids else tr).append(s)
    return tr, va


def read_image_bgr(path):
    """Membaca gambar dari disk dalam format BGR OpenCV."""
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(path)
    return img


def crop_with_bbox(image_bgr, bbox, pad=0.2):
    """Memotong gambar memakai bounding box dengan tambahan padding."""
    h, w = image_bgr.shape[:2]
    x, y, bw, bh = bbox.astype(np.float32)
    cx, cy = x + bw / 2.0, y + bh / 2.0
    bw2, bh2 = bw * (1.0 + pad), bh * (1.0 + pad)
    x1 = max(0, int(round(cx - bw2 / 2.0)))
    y1 = max(0, int(round(cy - bh2 / 2.0)))
    x2 = min(w, int(round(cx + bw2 / 2.0)))
    y2 = min(h, int(round(cy + bh2 / 2.0)))
    crop = image_bgr[y1:y2, x1:x2]
    return crop, np.array([x1, y1, x2 - x1, y2 - y1], dtype=np.float32)


def to_tensor_rgb(image_bgr, out_hw):
    """Mengubah gambar BGR menjadi tensor RGB yang sudah dinormalisasi."""
    h, w = out_hw
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (w, h), interpolation=cv2.INTER_LINEAR)
    x = resized.astype(np.float32) / 255.0
    x = np.transpose(x, (2, 0, 1))
    return torch.from_numpy(x)


def gaussian_heatmap(h, w, cx, cy, sigma=1.8):
    """Membuat heatmap Gaussian 2D dengan pusat di (cx, cy)."""
    ys = np.arange(h, dtype=np.float32)[:, None]
    xs = np.arange(w, dtype=np.float32)[None, :]
    g = np.exp(-((xs - cx) ** 2 + (ys - cy) ** 2) / (2.0 * sigma * sigma))
    return g.astype(np.float32)


def decode_argmax(hm):
    """Mengambil koordinat keypoint dari heatmap dengan argmax."""
    # hm: B,K,H,W -> B,K,2 ternormalisasi
    b, k, h, w = hm.shape
    flat = hm.view(b, k, -1)
    idx = flat.argmax(dim=-1)
    y = (idx // w).float() / max(h - 1, 1)
    x = (idx % w).float() / max(w - 1, 1)
    return torch.stack([x, y], dim=-1)


SKELETON_EDGES = [
    (0, 1), (0, 2),
    (1, 3), (2, 4),
    (5, 6),
    (5, 7), (7, 9),
    (6, 8), (8, 10),
    (11, 12),
    (5, 11), (6, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16),
]


def draw_pose(img_bgr, keypoints_xy, color_point=(0, 255, 0), color_line=(0, 200, 255)):
    """Menggambar skeleton COCO-17 di atas sebuah gambar."""
    out = img_bgr.copy()
    pts = np.asarray(keypoints_xy, dtype=np.float32)

    for a, b in SKELETON_EDGES:
        if a >= len(pts) or b >= len(pts):
            continue
        x1, y1 = pts[a]
        x2, y2 = pts[b]
        if x1 < 0 or y1 < 0 or x2 < 0 or y2 < 0:
            continue
        cv2.line(out, (int(x1), int(y1)), (int(x2), int(y2)), color_line, 2)

    for x, y in pts:
        if x < 0 or y < 0:
            continue
        cv2.circle(out, (int(x), int(y)), 3, color_point, -1)

    return out

Sel ini menyiapkan dataset, backbone model, dan metode forward untuk tiga mode HPE: top-down, bottom-up, dan regresi langsung.

In [ ]:
samples = load_samples(str(ANN_PATH), str(IMG_DIR), max_samples=MAX_SAMPLES)
train_samples, val_samples = split_samples(samples, val_ratio=0.2, seed=SEED)
print('total:', len(samples), 'train:', len(train_samples), 'val:', len(val_samples))

Sel ini memuat data ke dalam dataset lalu membagi data menjadi train dan validation.

In [ ]:
class ScratchPoseDataset(Dataset):
    """Pembungkus dataset untuk mode top-down, bottom-up, atau regresi."""

    def __init__(self, samples, mode='topdown'):
        """Menyimpan daftar sampel dan memilih mode pelatihan."""
        self.samples = samples
        self.mode = mode

    def __len__(self):
        """Mengembalikan jumlah sampel di dataset ini."""
        return len(self.samples)

    def __getitem__(self, idx):
        """Membaca satu sampel, melakukan praproses, dan membangun target latih."""
        s = self.samples[idx]
        img = read_image_bgr(s['image_path'])
        kpts = s['kpts'].copy()

        if self.mode in ('topdown', 'regression'):
            crop, crop_box = crop_with_bbox(img, s['bbox'], pad=0.2)
            if crop.size == 0:
                crop = img
                crop_box = np.array([0, 0, img.shape[1], img.shape[0]], dtype=np.float32)

            x = to_tensor_rgb(crop, (IN_H, IN_W)).float()
            kpts[:, 0] = (kpts[:, 0] - crop_box[0]) / max(crop_box[2], 1.0)
            kpts[:, 1] = (kpts[:, 1] - crop_box[1]) / max(crop_box[3], 1.0)
            kpts[:, 0] = np.clip(kpts[:, 0], 0.0, 1.0)
            kpts[:, 1] = np.clip(kpts[:, 1], 0.0, 1.0)
        else:
            h0, w0 = img.shape[:2]
            x = to_tensor_rgb(img, (IN_H, IN_W)).float()
            kpts[:, 0] = np.clip(kpts[:, 0] / max(w0, 1), 0.0, 1.0)
            kpts[:, 1] = np.clip(kpts[:, 1] / max(h0, 1), 0.0, 1.0)

        vis = (kpts[:, 2] > 0).astype(np.float32)

        if self.mode == 'topdown':
            hm = np.zeros((17, HM_H, HM_W), dtype=np.float32)
            for j in range(17):
                if vis[j] <= 0:
                    continue
                cx = float(kpts[j, 0] * (HM_W - 1))
                cy = float(kpts[j, 1] * (HM_H - 1))
                hm[j] = gaussian_heatmap(HM_H, HM_W, cx, cy)
            return x, torch.from_numpy(hm), torch.from_numpy(kpts[:, :2].astype(np.float32)), torch.from_numpy(vis)

        if self.mode == 'bottomup':
            hm_kpt = np.zeros((17, HM_H, HM_W), dtype=np.float32)
            for j in range(17):
                if vis[j] <= 0:
                    continue
                cx = float(kpts[j, 0] * (HM_W - 1))
                cy = float(kpts[j, 1] * (HM_H - 1))
                hm_kpt[j] = gaussian_heatmap(HM_H, HM_W, cx, cy)

            center = np.zeros((1, HM_H, HM_W), dtype=np.float32)
            vis_ids = np.where(vis > 0)[0]
            if len(vis_ids) > 0:
                cx = float(np.mean(kpts[vis_ids, 0]) * (HM_W - 1))
                cy = float(np.mean(kpts[vis_ids, 1]) * (HM_H - 1))
                center[0] = gaussian_heatmap(HM_H, HM_W, cx, cy, sigma=2.2)

            target = np.concatenate([hm_kpt, center], axis=0)
            return x, torch.from_numpy(target), torch.from_numpy(kpts[:, :2].astype(np.float32)), torch.from_numpy(vis)

        # regresi langsung
        return x, torch.from_numpy(kpts[:, :2].astype(np.float32)), torch.from_numpy(kpts[:, :2].astype(np.float32)), torch.from_numpy(vis)


class TinyTopDown(nn.Module):
    """CNN minimal untuk memprediksi satu heatmap per keypoint."""

    def __init__(self):
        """Membangun backbone konvolusional ringan dan head heatmap."""
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.Conv2d(128, 17, 1),
        )

    def forward(self, x):
        """Mengembalikan heatmap COCO-17 untuk satu batch input."""
        return self.net(x)


class TinyBottomUp(nn.Module):
    """CNN minimal untuk memprediksi heatmap keypoint dan heatmap pusat."""

    def __init__(self):
        """Membangun prediktor heatmap penuh gambar yang ringan."""
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.Conv2d(128, 18, 1),
        )

    def forward(self, x):
        """Mengembalikan 17 peta keypoint ditambah 1 peta pusat."""
        return self.net(x)


class TinyRegressor(nn.Module):
    """CNN minimal yang memprediksi koordinat keypoint ternormalisasi."""

    def __init__(self):
        """Membangun backbone dan head fully connected untuk regresi."""
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(128, 128), nn.ReLU(True), nn.Linear(128, 34))

    def forward(self, x):
        """Mengembalikan 17 koordinat keypoint dalam bentuk (x, y) ternormalisasi."""
        y = self.fc(self.backbone(x)).view(-1, 17, 2)
        return torch.sigmoid(y)

Sel ini membentuk model sesuai mode yang dipilih, lalu menjalankan training singkat dan menyimpan checkpoint.

In [ ]:
train_ds = ScratchPoseDataset(train_samples, mode=MODEL_KIND)
val_ds = ScratchPoseDataset(val_samples, mode=MODEL_KIND)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

if MODEL_KIND == 'topdown':
    model = TinyTopDown()
elif MODEL_KIND == 'bottomup':
    model = TinyBottomUp()
else:
    model = TinyRegressor()

model = model.to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    model.train()
    running = 0.0

    for x, target, gt_xy, vis in train_dl:
        x = x.to(device)
        target = target.to(device)
        vis = vis.to(device)

        pred = model(x)

        if MODEL_KIND in ('topdown', 'bottomup'):
            loss = F.mse_loss(pred, target)
        else:
            l1 = torch.abs(pred - target).sum(dim=-1)
            loss = (l1 * vis).sum() / vis.sum().clamp(min=1.0)

        opt.zero_grad()
        loss.backward()
        opt.step()
        running += float(loss.item())

    print(f'epoch={epoch+1} loss={running/max(1,len(train_dl)):.5f}')

torch.save(model.state_dict(), f'/kaggle/working/scratch_{MODEL_KIND}.pt')
print('saved model:', f'/kaggle/working/scratch_{MODEL_KIND}.pt')

Sel ini menjalankan inferensi pada satu contoh gambar dan menampilkan hasil skeleton prediksi di atas citra.

In [ ]:
# Coba inferensi pada satu gambar demo
model.eval()

if SAMPLE_IMAGE_PATH:
    demo_img_bgr = read_image_bgr(SAMPLE_IMAGE_PATH)
    demo_rgb = cv2.cvtColor(demo_img_bgr, cv2.COLOR_BGR2RGB)
    x = to_tensor_rgb(demo_img_bgr, (IN_H, IN_W)).float()
    demo_label = f'path: {Path(SAMPLE_IMAGE_PATH).name}'
    sample_idx = None
else:
    if SAMPLE_MODE == 'random':
        sample_idx = random.randrange(len(val_ds))
    else:
        sample_idx = int(SAMPLE_IDX) % max(1, len(val_ds))

    x, _, _, _ = val_ds[sample_idx]
    demo_rgb = (np.transpose(x.numpy(), (1, 2, 0)) * 255.0).astype(np.uint8)
    demo_img_bgr = cv2.cvtColor(demo_rgb, cv2.COLOR_RGB2BGR)
    demo_label = f'{SAMPLE_MODE}: {sample_idx}'

inp = x.unsqueeze(0).to(device)

with torch.no_grad():
    out = model(inp)

if MODEL_KIND == 'topdown':
    pred_xy = decode_argmax(out).cpu().numpy()[0]
elif MODEL_KIND == 'bottomup':
    pred_xy = decode_argmax(out[:, :17]).cpu().numpy()[0]
else:
    pred_xy = out.cpu().numpy()[0]

# Ubah ke koordinat piksel pada ukuran input model
pred_xy_px = np.zeros_like(pred_xy)
pred_xy_px[:, 0] = pred_xy[:, 0] * (IN_W - 1)
pred_xy_px[:, 1] = pred_xy[:, 1] * (IN_H - 1)

base_bgr = cv2.resize(demo_img_bgr, (IN_W, IN_H), interpolation=cv2.INTER_LINEAR)
out_bgr = draw_pose(base_bgr, pred_xy_px)
img_rgb = cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(5, 7))
plt.imshow(img_rgb)
plt.title(f'Prediction demo ({MODEL_KIND})')
plt.axis('off')
plt.show()

print('Demo source:', demo_label)

Sel ini mengatur contoh gambar untuk inferensi, lalu memvisualisasikan skeleton prediksi dari model yang sudah dilatih.